# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **what are the fastest closure times for City 311 tickets**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-09-09 17:43:10 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-09-09T17:43:10.035232")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for '311 service requests closure time':

1. **NYPD Compstat Data 2019** (19.4% match)
   Resource: Volume 26 Week 11
   Resource ID: `cc560dbf-96f5-4ce5-8a0f-a5aa9a6bc667`
   Dataset ID: `027bee88-b3f6-4b05-999c-0e1d4a826ae9`
   Format: CSV | Rows: 1,275 | Cols: 20
   Tags: NYPD, CompStat, Crime Statistics, New York City, 2019, Weekly Crime Data, Felony Assault, Burglary, Robbery, Precinct Data
   This dataset provides weekly CompStat data from the NYPD for the week of March 11-17, 2019. It details crime statistics across various categories, including 
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": '311 service requests closure time', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `311 service requests`

**Result preview:**
```
Found 8 datasets matching '311 service requests'

1. **211 Service Requests**
   ID: `211-requests`
   ## Overview  Every day, thousands of people in our region are struggling to put food on the table, keep the lights on, and keep a roof over their head. 2-1-1 is a 24/7 telephone 
   - 2-1-1 Requests/Referrals (2023-2025) from Western Pennsylvania (CSV) [DataStore] ID: `0ce22487-5707-4e64-8e59-5b189cfc69cb`
   - 2-1-1 Requests (2020-2023) (CSV) [DataStore] ID: `94f52ddb-acf3-4728-93b9-4544988f1d6f`
   - 211 Counts Dashboard (HTML) ID: `3cf2e456-91a7-4958-8a2d-0f40725d3fc3`

2. **311 Data Arc
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": '311 service requests', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 967,237
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject     created_date_et    created_date_utc last_modified_date_et last
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 4: Load Data from Resource

**Resource ID:** `29462525-62a6-45bf-9b5e-ad2e1c06348d`
**Limit:** 5

**Result preview:**
```
Resource: 29462525-62a6-45bf-9b5e-ad2e1c06348d
Total records: 815,417
Loaded: 5
Fields (28): group_id, num_requests, parent_closed, status_name, status_code, dept, request_type_name, request_type_id, create_date_et, create_date_utc, last_action_et, last_action_utc, closed_date_et, closed_date_utc, origin, street, cross_street, street_id, cross_street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy

Sample (5 rows):

group_id  num_requests  parent_closed status_name  status_code                 dept request_type_name request_type_id  
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '29462525-62a6-45bf-9b5e-ad2e1c06348d', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 5: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT subject, COUNT(*) as cnt,
       AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours
FROM "5202679a-d243-402e-b82a-63189995a942"
WHERE status = 'Closed' AND closed_date_et IS NOT NULL AND closed_date_et != '' 
  AND created_date_et IS NOT NULL AND created_date_et != ''
GROUP BY subject
HAVING COUNT(*) >= 30
ORDER BY avg_hours ASC
LIMIT 15
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 5: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT subject, COUNT(*) as cnt,\n       AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours\nFROM "5202679a-d243-402e-b82a-63189995a942"\nWHERE status = \'Closed\' AND closed_date_et IS NOT NULL AND closed_date_et != \'\' \n  AND created_date_et IS NOT NULL AND created_date_et != \'\'\nGROUP BY subject\nHAVING COUNT(*) >= 30\nORDER BY avg_hours ASC\nLIMIT 15'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 6: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT subject, COUNT(*) as cnt, AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = 'Closed' AND closed_date_et IS NOT NULL AND closed_date_et != '' GROUP BY subject HAVING COUNT(*) >= 30 ORDER BY avg_hours ASC LIMIT 15
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 6: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT subject, COUNT(*) as cnt, AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = \'Closed\' AND closed_date_et IS NOT NULL AND closed_date_et != \'\' GROUP BY subject HAVING COUNT(*) >= 30 ORDER BY avg_hours ASC LIMIT 15'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 7: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT COUNT(*) FROM "5202679a-d243-402e-b82a-63189995a942"
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 7: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT COUNT(*) FROM "5202679a-d243-402e-b82a-63189995a942"'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 8: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT COUNT(*) as cnt FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status='Closed'
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 8: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT COUNT(*) as cnt FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status=\'Closed\''
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 9: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT status, count(*) FROM "5202679a-d243-402e-b82a-63189995a942" GROUP BY status
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 9: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT status, count(*) FROM "5202679a-d243-402e-b82a-63189995a942" GROUP BY status'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 10: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT 1
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 10: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT 1'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 11: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT COUNT(*) FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d"
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 11: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT COUNT(*) FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d"'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 12: SQL Analysis Query

**SQL:**
```sql
SELECT 1
```

**Result preview:**
```
SQL: SELECT 1 LIMIT 10000
Rows: 1
Columns: ?column?

 ?column?
        1
```


In [ ]:
# Step 12: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT 1'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 13: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 500
**Filters:** `{"status": "Closed"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 865,658
Loaded: 500
Fields (5): case_number, status, subject, created_date_et, closed_date_et

Sample (15 rows):

case_number status                                  subject     created_date_et      closed_date_et
25-00100407 Closed                    City Tree Maintenance 2025-10-21T16:01:11 2026-09-09T07:56:28
26-00044395 Closed                                 Potholes 2026-04-17T07:06:27 2026-09-09T07:55:38
26-00090385 Closed                    City Tree Maintenance 2026-08-25T17:27:00 2026-09-09T07:54:25
26-00043315 Closed      
```


In [ ]:
# Step 13: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 500, "filters": {"status": "Closed"}, "sort": 'closed_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 14: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT subject, COUNT(*) as cnt, AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = 'Closed' AND closed_date_et IS NOT NULL AND closed_date_et != '' GROUP BY subject HAVING COUNT(*) >= 30 ORDER BY avg_hours ASC LIMIT 15
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 14: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT subject, COUNT(*) as cnt, AVG(EXTRACT(EPOCH FROM (CAST(closed_date_et AS timestamp) - CAST(created_date_et AS timestamp)))/3600.0) as avg_hours FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = \'Closed\' AND closed_date_et IS NOT NULL AND closed_date_et != \'\' GROUP BY subject HAVING COUNT(*) >= 30 ORDER BY avg_hours ASC LIMIT 15'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 15: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT subject, COUNT(*) as cnt FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = 'Closed' GROUP BY subject ORDER BY cnt DESC LIMIT 10
```

**Result preview:**
```
SQL error (HTTP 403): <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again
```


In [ ]:
# Step 15: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT subject, COUNT(*) as cnt FROM "5202679a-d243-402e-b82a-63189995a942" WHERE status = \'Closed\' GROUP BY subject ORDER BY cnt DESC LIMIT 10'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **I wasn't able to complete the analysis for this question. It may help to rephrase it — for example:

- Name a specific place (a city, county, or state)
- Name a specific time period (a year or range of years)
- Ask about one metric at a time

You can also ask *"What datasets are available on this topic?"* to see what I can search.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-09-09

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-09-09 17:43:10
- **Query**: what are the fastest closure times for City 311 tickets
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
